In [24]:
# Import the below libraries:
"""For faster loading, consider importing the libraries in separate cells."""
# To create reproducible file paths
import os # To interact with the operating system
from pathlib import Path # To create Path objects
import pathlib # To access the object-oriented library for paths

# For unzipping folders
import time # To handle time-related tasks
import zipfile # To download and extract zip files

# For saving Python objects
import pickle # To pickle datasets that take long periods to process

# To use APIs
import earthaccess # For logging into NASA Earth Access
import requests # For sending API requests

# To download Global Biodiversity Information Facility (GBIF) data
from getpass import getpass # To obtain a GBIF login and password
from pygbif import species
import pygbif.occurrences as occ # To download species occurrence data from GBIF
import pygbif.species as species # To identify specific species datasets to download on GBIF

# To download National Ecological Observatory Network (NEON) soil data
import neonutilities as nu # To access and download NEON data

# To work with different types of data
import geopandas as gpd # To make GeoDataFrames/work with vector data
from glob import glob # To combine data arrays
from math import floor, ceil # For dealing with integers
import numpy as np # To work with arrays
import pandas as pd # To work with dataframes
import rasterio # For handling raster soil maps
import rioxarray as rxr # To work with raster data
import rioxarray.merge as rxrm # To merge raster data
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries
import xarray as xr # To use xarray datasets
import xrspatial # To handle spatial data

# For visualization and interactive plotting
import holoviews as hv # To create interactive hvPlots
import hvplot.pandas # To enable hvPlot interactive plotting for Pandas dataframes
import hvplot.xarray # To enable hvPlot interactive plotting for xarray datasets
import matplotlib.gridspec as gridspec # To display the plot legends properly
import matplotlib.pyplot as plt # To import the Pyplot module

In [25]:
# Create a designated folder for the GBIF and soil data
"""Note: the precise destination of the repository folder will vary from individual to individual."""
soil_nutrient_data_dir = os.path.join(
    pathlib.Path.home(),
    # In the earth-analytics data folder
    'earth-analytics',
    'data',
    # Specify the destination as inside the "PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation" repository
    'PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation',
    'preferred_soil_characteristics_for_species'
)

# Create the directory
os.makedirs(soil_nutrient_data_dir, exist_ok=True)

In [26]:
# Create a directory to store the GBIF data
gbif_dir = os.path.join(soil_nutrient_data_dir, 'gbif_PLANT-B_dir')
os.makedirs(gbif_dir, exist_ok=True)

In [27]:
# Permanently log into GBIF
# Do not reset credentials to avoid repeated login requests
reset_credentials = False

# Request and store username
if (not ('GBIF_USER'  in os.environ)) or reset_credentials:
    os.environ['GBIF_USER'] = input('GBIF username:')

# Request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset_credentials:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
# Request and store email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset_credentials:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [28]:
# Check that the login attempt was successful
'GBIF_PWD' in os.environ

True

In [29]:
# Create a list of the target species
species_list = [
    # Dwarf white isopod
    'Trichorhina tomentosa',
    # Temperate white springtail
    'Folsomia candida',
    # Spreading earthmoss
    'Physcomitrella patens'
]

# Pull the species info from GBIF
species_info = species.name_lookup(species_list, rank = 'Species')

# Grab the result for the white dwarf isopod and print it
isopod_result = species_info['results'][0]
print("\nWhite dwarf isopod (Trichorhina tomentosa):")
print(isopod_result)

# Grab the result for the temperate white springtail and print it
springtail_result = species_info['results'][1]
print("\nTemperate white springtail (Folsomia candida):")
print(springtail_result)

# Grab the result for the spreading earthmoss and print it
moss_result = species_info['results'][2]
print("\nSpreading earthmoss (Physcomitrella patens):")
print(moss_result)


White dwarf isopod (Trichorhina tomentosa):
{'key': 104276027, 'datasetKey': 'fab88965-e69d-4491-a04d-e3198b626e52', 'parentKey': 104276025, 'parent': 'Trichorhina', 'kingdom': 'Metazoa', 'phylum': 'Arthropoda', 'order': 'Isopoda', 'family': 'Platyarthridae', 'genus': 'Trichorhina', 'species': 'Trichorhina tomentosa', 'kingdomKey': 103832354, 'phylumKey': 104183026, 'classKey': 104244551, 'orderKey': 104273982, 'familyKey': 104276010, 'genusKey': 104276025, 'speciesKey': 104276027, 'scientificName': 'Trichorhina tomentosa', 'canonicalName': 'Trichorhina tomentosa', 'nameType': 'SCIENTIFIC', 'taxonomicStatus': 'ACCEPTED', 'rank': 'SPECIES', 'origin': 'SOURCE', 'numDescendants': 0, 'numOccurrences': 0, 'taxonID': '172010', 'habitats': [], 'nomenclaturalStatus': [], 'threatStatuses': [], 'descriptions': [], 'vernacularNames': [], 'synonym': False, 'higherClassificationMap': {'103832354': 'Metazoa', '104183026': 'Arthropoda', '104244551': 'Malacostraca', '104273982': 'Isopoda', '104276010

In [30]:
# Get the species keys for all three species 
species_keys = [
    isopod_result['key'], 
    springtail_result['key'], 
    moss_result['key']]

# Print all three keys
species_keys

[104276027, 145956763, 2204981]

In [31]:
# Create the file path
gbif_pattern = os.path.join(gbif_dir, '*.csv')

# Create a function to download the occurrence data for all three species:
# Iterate over the species keys to download data for each species
for key in species_keys:

    # Check if the data for the species has not already been downloaded
    if not glob(gbif_pattern):
        # Use the "key" field to submit the query for each species
        gbif_query = occ.download([
            f"speciesKey = {key}", 
            "hasCoordinate = True", # Only download an observation if it has coordinates
        ])

        # Only download the data for each species once
        if not 'GBIF_DOWNLOAD_KEY' in os.environ:
            os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0] # Store download key in environment
            download_key = os.environ['GBIF_DOWNLOAD_KEY']
            time.sleep(5)
        # Download the data
        download_info = occ.download_get(
            download_key, # Call the download key
            path = gbif_dir # Store the data directly in gbif_dir
        )
            
        # Unzip the file
        with zipfile.ZipFile(download_info['path']) as download_zip:
            download_zip.extractall(path = gbif_dir)

# Locate the CSV file path for the downloaded data
gbif_path = glob(gbif_pattern)[0]
print(f"Downloaded the occurrence data for all three species: {gbif_path}")

Downloaded the occurrence data for all three species: C:\Users\livth\earth-analytics\data\PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation\preferred_soil_characteristics_for_species\gbif_PLANT-B_dir\0080604-260226173443078.csv


In [32]:
# Look at the download information
occ.download_meta("0080604-260226173443078") # Input the download key to view the information

{'key': '0080604-260226173443078',
 'doi': '10.15468/dl.hcmxe3',
 'license': 'http://creativecommons.org/licenses/by-nc/4.0/legalcode',
 'request': {'predicate': {'type': 'in',
   'key': 'TAXON_KEY',
   'values': ['2120383', '2204981', '2674024'],
   'matchCase': False},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-04-03T02:14:21.475+00:00',
 'modified': '2026-04-03T02:15:59.713+00:00',
 'eraseAfter': '2026-10-03T02:14:21.405+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0080604-260226173443078.zip',
 'size': 216460,
 'totalRecords': 3529,
 'numberDatasets': 265}

In [33]:
# Define the path to the CSV
"""This is a temporary fix until the download function can be troubleshooted."""
csv_file_path = os.path.join(gbif_dir, "0080604-260226173443078.csv")

# Read the CSV
gbif_df = pd.read_csv(
    csv_file_path,
    delimiter = '\t'
)

# Check the dataframe
gbif_df

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,92633894,85042096-f762-11e1-a439-00145eb45e9a,NaN,Plantae,Bryophyta,Bryopsida,Funariales,Funariaceae,Physcomitrium,Physcomitrium patens,...,J.-P. Frahm,NaN,CC_BY_NC_4_0,NaN,J.-P. Frahm,NaN,NaN,2026-03-27T10:42:08.410Z,NaN,CONTINENT_DERIVED_FROM_COUNTRY;BASIS_OF_RECORD...
1,920117266,f02240a8-f206-467b-a379-fdce36beb808,97734,Animalia,Arthropoda,Collembola,Entomobryomorpha,Isotomidae,Folsomia,Folsomia candida,...,NaN,NaN,CC_BY_NC_4_0,NaN,NaN,NaN,NaN,2026-03-03T19:34:04.452Z,NaN,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...
2,920117240,f02240a8-f206-467b-a379-fdce36beb808,97733,Animalia,Arthropoda,Collembola,Entomobryomorpha,Isotomidae,Folsomia,Folsomia candida,...,NaN,NaN,CC_BY_NC_4_0,NaN,NaN,NaN,NaN,2026-03-03T19:34:04.452Z,NaN,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...
3,920117215,f02240a8-f206-467b-a379-fdce36beb808,97732,Animalia,Arthropoda,Collembola,Entomobryomorpha,Isotomidae,Folsomia,Folsomia candida,...,NaN,NaN,CC_BY_NC_4_0,NaN,NaN,NaN,NaN,2026-03-03T19:34:04.453Z,NaN,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...
4,920117190,f02240a8-f206-467b-a379-fdce36beb808,97731,Animalia,Arthropoda,Collembola,Entomobryomorpha,Isotomidae,Folsomia,Folsomia candida,...,NaN,NaN,CC_BY_NC_4_0,NaN,NaN,NaN,NaN,2026-03-03T19:34:04.453Z,NaN,COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDI...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3524,1019473312,fea77396-7b26-44cb-8580-aefadc8db276,http://coldb.mnhn.fr/catalognumber/mnhn/pc/fus...,Plantae,Bryophyta,Bryopsida,Funariales,Funariaceae,Physcomitrium,Physcomitrium patens,...,NaN,NaN,CC_BY_4_0,NaN,"Bizot, M.",NaN,NaN,2025-11-25T18:39:16.546Z,NaN,NaN
3525,1019473265,fea77396-7b26-44cb-8580-aefadc8db276,http://coldb.mnhn.fr/catalognumber/mnhn/pc/fus...,Plantae,Bryophyta,Bryopsida,Funariales,Funariaceae,Physcomitrium,Physcomitrium patens,...,NaN,NaN,CC_BY_4_0,NaN,"Parriat, H.",NaN,NaN,2025-11-25T18:39:17.202Z,NaN,NaN
3526,1019473258,fea77396-7b26-44cb-8580-aefadc8db276,http://coldb.mnhn.fr/catalognumber/mnhn/pc/fus...,Plantae,Bryophyta,Bryopsida,Funariales,Funariaceae,Physcomitrium,Physcomitrium patens,...,NaN,NaN,CC_BY_4_0,NaN,"Bizot, M.",NaN,NaN,2025-11-25T18:39:10.117Z,NaN,NaN
3527,1019473243,fea77396-7b26-44cb-8580-aefadc8db276,http://coldb.mnhn.fr/catalognumber/mnhn/pc/fus...,Plantae,Bryophyta,Bryopsida,Funariales,Funariaceae,Physcomitrium,Physcomitrium patens,...,NaN,NaN,CC_BY_4_0,NaN,"Duclos, P.",NaN,NaN,2025-11-25T18:39:17.601Z,NaN,NaN


In [34]:
# Look at the columns
gbif_df.columns

Index(['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'species', 'infraspecificEpithet',
       'taxonRank', 'scientificName', 'verbatimScientificName',
       'verbatimScientificNameAuthorship', 'countryCode', 'locality',
       'stateProvince', 'occurrenceStatus', 'individualCount',
       'publishingOrgKey', 'decimalLatitude', 'decimalLongitude',
       'coordinateUncertaintyInMeters', 'coordinatePrecision', 'elevation',
       'elevationAccuracy', 'depth', 'depthAccuracy', 'eventDate', 'day',
       'month', 'year', 'taxonKey', 'speciesKey', 'basisOfRecord',
       'institutionCode', 'collectionCode', 'catalogNumber', 'recordNumber',
       'identifiedBy', 'dateIdentified', 'license', 'rightsHolder',
       'recordedBy', 'typeStatus', 'establishmentMeans', 'lastInterpreted',
       'mediaType', 'issue'],
      dtype='str')

In [35]:
# Specify the columns to keep
"""Only columns related to the species name, geometry, elevation, and observation date will be kept."""
columns_to_keep = [
    'species', 'decimalLatitude', 'decimalLongitude', 'elevation', 'eventDate'
]

# Filter the dataset to keep only the necessary columns
gbif_filtered_df = gbif_df[columns_to_keep]

# Remove observations that do not have coordinates
clean_gbif_gdf = gbif_filtered_df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

# Convert the filtered dataframe into a GeoDataFrame (GDF)
gbif_gdf = gpd.GeoDataFrame(
    clean_gbif_gdf,
    # Add geometry columns to convert to a GDF
    geometry = gpd.points_from_xy(
        clean_gbif_gdf.decimalLongitude,
        clean_gbif_gdf.decimalLatitude
    ),
    crs = 'EPSG:4326'  # Set the Coordinate Reference System (CRS)
)

# Group all spreading earthmoss sightings under one scientific name
gbif_gdf['species'] = gbif_gdf['species'].replace({'Physcomitrella patens': 'Physcomitrium patens'})

# Display the GDF data
gbif_gdf

,species,decimalLatitude,decimalLongitude,elevation,eventDate,geometry
1,Folsomia candida,51.970641,5.641871,NaN,1986-10-31,POINT (5.64187 51.97064)
2,Folsomia candida,51.970641,5.641871,NaN,1986-10-31,POINT (5.64187 51.97064)
3,Folsomia candida,51.970641,5.641871,NaN,1986-10-31,POINT (5.64187 51.97064)
4,Folsomia candida,51.970641,5.641871,NaN,1986-10-31,POINT (5.64187 51.97064)
5,Folsomia candida,52.410727,6.893704,NaN,2000-05-30,POINT (6.8937 52.41073)
...,...,...,...,...,...,...
3511,Physcomitrium patens,59.868390,17.634420,NaN,1873-09,POINT (17.63442 59.86839)
3512,Physcomitrium patens,59.868390,17.634420,NaN,1873-09,POINT (17.63442 59.86839)
3513,Physcomitrium patens,60.251740,17.911430,NaN,NaN,POINT (17.91143 60.25174)
3514,Physcomitrium patens,59.868390,17.634420,NaN,1855-11,POINT (17.63442 59.86839)


In [36]:
# Plot the points to ensure they were processed properly
species_distribution_plot = gbif_gdf.hvplot(
    c="species",  # Color the points so they are identified by species
    cmap="tab20",  # Set the color map
    crs="EPSG:4326", # Set the CRS to ESPG:4326
    # Set the width and height
    frame_height=400,
    frame_width=650,
    geo=True,
    hover_cols=["species"], # Make the species visible when a point is hovered over
    tiles="CartoLight",
    title=f'Distribution of PLANT-B Species', # Set the title
    x="decimalLongitude", # Define the x-axis by the longitude
    y="decimalLatitude", # Define the y-axis by the latitude 
)
species_distribution_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [decimalLongitude,decimalLatitude]   (species)

In [37]:
# Create a path to save the species distribution plot
species_distribution_plot_path = Path('PLANT-B_Species_Distribution_Plot.html')

# Save the plot as an interactive HTML
if not species_distribution_plot_path.exists():
    hvplot.save(species_distribution_plot, 'PLANT-B_Species_Distribution_Plot.html')
else:
    print("The interactive plot showing the distribution of PLANT-B's chosen species has already been saved.")

The interactive plot showing the distribution of PLANT-B's chosen species has already been saved.
